# Paper Evaluations — CIs, Calibration, Robustness, Per-Generator, LOGO

Run **after** the training notebooks. Loads the best MFFT checkpoint and produces
the remaining tables the paper needs (Section VIII items 2-4, 7):

1. Bootstrap 95% confidence intervals + McNemar test
2. Temperature scaling (pre/post ECE, Brier)
3. Robustness suite (JPEG / downscale / blur)
4. Per-generator accuracy breakdown
5. Leave-one-generator-out (LOGO) train/test manifests

No GPU -> SMOKE mode: small subsets, one LOGO generator, still exercises every code path.

In [ ]:
# Cell 0: Clone repo from GitHub (Kaggle needs this — local files aren’t available)
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/MIHMahmudEli/ai-image-detection-research.git"
CLONE_DIR = Path("/kaggle/working/ai-image-detection-research")

if not CLONE_DIR.exists():
    print(f"Cloning repo from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
    print("Clone complete")
else:
    print("Repo already cloned")

# Set up paths so all imports work
import os
os.chdir("/kaggle/working")
sys.path.insert(0, str(CLONE_DIR / "model"))
print(f"Project root: {CLONE_DIR}")

In [ ]:
# Cell 1: Imports & Environment Setup (Kaggle/DGX/Local auto-detect)
import os, sys, math, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm

# ── Kaggle / DGX / Local auto-detection ──
sys.path.insert(0, str(Path.cwd().resolve() / 'ai-image-detection-research' / 'model'))
from src.kaggle_utils import KaggleEnv
env = KaggleEnv(project_root_search=True)
PROJECT_ROOT = env.project_root
os.chdir(env.working_dir)

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMOKE_TEST = not torch.cuda.is_available()
print(f'Device: {device} | SMOKE_TEST: {SMOKE_TEST}')

In [ ]:
# Cell 2: Config, shared split, output dir
from src.dataset import create_split_dataloaders
from src.model import build_mfft, count_parameters

VARIANT = 'base'
IMAGE_SIZE = 224 if SMOKE_TEST else 384
BATCH = 8 if SMOKE_TEST else 64
MAX_SAMPLES = 600 if SMOKE_TEST else None

# ── Manifest: local → HF → rebuild from Kaggle mounts ──
_manifest = PROJECT_ROOT / 'dataset' / 'metadata' / 'train_manifest.csv'
if not _manifest.exists() or _manifest.stat().st_size < 1000:
    env.download_manifest(_manifest)
if not _manifest.exists() or _manifest.stat().st_size < 1000:
    env.rebuild_manifest_from_kaggle(_manifest)
if not _manifest.exists():
    _manifest = PROJECT_ROOT / 'dataset' / 'metadata' / 'clean_metadata.csv'
    print('WARNING: using clean_metadata.csv (run rebuild_manifest.py for full scale)')
MANIFEST = str(_manifest)

_split_name = 'split_indices_smoke.json' if SMOKE_TEST else 'split_indices.json'
train_loader, val_loader, test_loader = create_split_dataloaders(
    root_dir=str(PROJECT_ROOT), metadata_paths=[MANIFEST],
    batch_size=BATCH, num_workers=0 if SMOKE_TEST else 8, size=IMAGE_SIZE,
    val_split=0.10, test_split=0.10, seed=SEED, use_weighted_sampler=False,
    split_index_path=str(PROJECT_ROOT / 'dataset' / 'metadata' / _split_name),
    max_samples=MAX_SAMPLES,
)
test_dataset = test_loader.dataset

OUT = PROJECT_ROOT / 'paper' / 'result' / ('verify' if SMOKE_TEST else 'full_scale') / 'paper_evals'
OUT.mkdir(parents=True, exist_ok=True)
print(f'Outputs -> {OUT}')

In [ ]:
# Cell 3: Load best checkpoint (searches known locations)
def load_variant(variant):
    model = build_mfft(variant)
    candidates = [
        PROJECT_ROOT / 'model' / 'checkpoints' / f'{variant}_model' / f'best_mfft_{variant}.pt',
        PROJECT_ROOT / 'model' / 'checkpoints' / 'verify' / f'{variant}_model' / 'best.pt',
        PROJECT_ROOT / 'model' / 'checkpoints' / 'test' / f'{variant}_model' / 'best.pt',
        PROJECT_ROOT / 'model' / 'checkpoints' / f'best_mfft_{variant}.pt',
    ]
    for ck in candidates:
        if ck.exists():
            try:
                model.load_state_dict(torch.load(ck, map_location='cpu'), strict=True)
                print(f'{variant}: loaded {ck.relative_to(PROJECT_ROOT)}')
                return model.to(device).eval(), str(ck)
            except Exception as e:
                print(f'{variant}: {ck.name} incompatible ({e}); trying next')
    print(f'{variant}: NO checkpoint found - using random init (pipeline check only)')
    return model.to(device).eval(), None

model, ckpt_path = load_variant(VARIANT)
print(f'Params: {count_parameters(model):,}')

In [ ]:
# Cell 4: Collect logits on val (for temperature fitting) and test
@torch.no_grad()
def collect(loader, net):
    logits_all, labels_all = [], []
    for images, labels in loader:
        logits_all.append(net(images.to(device)).cpu())
        labels_all.append(labels)
    return torch.cat(logits_all), torch.cat(labels_all)

val_logits, val_labels = collect(val_loader, model)
test_logits, test_labels = collect(test_loader, model)
y_true = test_labels.numpy()
y_score = F.softmax(test_logits, dim=-1)[:, 1].numpy()
y_pred = (y_score >= 0.5).astype(int)
acc = (y_pred == y_true).mean() * 100
print(f'Test: n={len(y_true)}, acc={acc:.2f}%')

In [ ]:
# Cell 5: Bootstrap 95% confidence intervals
from src.stats import bootstrap_ci

rows = []
for metric, arg in [('accuracy', y_pred), ('precision', y_pred),
                    ('recall', y_pred), ('f1', y_pred), ('auc', y_score)]:
    ci = bootstrap_ci(y_true, arg, metric=metric, n_resamples=1000, seed=SEED)
    rows.append({'metric': metric, 'point': round(ci['point'], 4),
                 'ci_lower': round(ci['lower'], 4), 'ci_upper': round(ci['upper'], 4)})
    print(f"{metric:>10}: {ci['point']:.4f}  [{ci['lower']:.4f}, {ci['upper']:.4f}]")
pd.DataFrame(rows).to_csv(OUT / f'{VARIANT}_metrics_ci.csv', index=False)
print('saved', OUT / f'{VARIANT}_metrics_ci.csv')

In [ ]:
# Cell 6: Temperature scaling (fit on val, report pre/post on test)
from src.calibration import TemperatureScaler, expected_calibration_error, brier_score

pre_ece = expected_calibration_error(y_true, y_score)
pre_brier = brier_score(y_true, y_score)

scaler = TemperatureScaler()
T = scaler.fit(val_logits, val_labels)
cal_score = scaler.calibrate(test_logits)[:, 1].numpy()
post_ece = expected_calibration_error(y_true, cal_score)
post_brier = brier_score(y_true, cal_score)

print(f'Temperature: {T:.3f}')
print(f'ECE   pre={pre_ece:.4f}  post={post_ece:.4f}')
print(f'Brier pre={pre_brier:.4f}  post={post_brier:.4f}')
pd.DataFrame([{'temperature': round(T, 4),
               'ece_pre': round(pre_ece, 4), 'ece_post': round(post_ece, 4),
               'brier_pre': round(pre_brier, 4), 'brier_post': round(post_brier, 4)}]
             ).to_csv(OUT / f'{VARIANT}_calibration.csv', index=False)
print('saved', OUT / f'{VARIANT}_calibration.csv')

In [ ]:
# Cell 7: McNemar test vs a second model (tiny checkpoint if available)
from src.stats import mcnemar_test

other, other_ck = load_variant('tiny')
if other_ck is None:
    print('No second checkpoint - McNemar demo runs against tiny random init')
other_logits, _ = collect(test_loader, other)
other_pred = other_logits.argmax(dim=-1).numpy()

res = mcnemar_test(y_true, y_pred, other_pred)
print(res)
pd.DataFrame([{'model_a': VARIANT, 'model_b': 'tiny', **res}]
             ).to_csv(OUT / 'mcnemar.csv', index=False)
print('saved', OUT / 'mcnemar.csv')
# NOTE (full scale): also compare against the strongest baseline by loading its
# checkpoint from model/checkpoints/<baseline>/best.pt with its class from src.baselines.

In [ ]:
# Cell 8: Robustness suite (JPEG / downscale / blur)
from src.robustness import run_robustness_suite

rob = run_robustness_suite(
    model, test_dataset, device=str(device), size=IMAGE_SIZE,
    batch_size=BATCH, max_samples=120 if SMOKE_TEST else 2000,
    out_csv=str(OUT / f'{VARIANT}_robustness.csv'),
)

In [ ]:
# Cell 9: Per-generator accuracy breakdown
from src.logo_eval import evaluate_per_generator

per_gen = evaluate_per_generator(
    model, MANIFEST, images_root=str(PROJECT_ROOT / 'dataset' / 'images'),
    device=str(device), size=IMAGE_SIZE, batch_size=BATCH,
    max_per_generator=40 if SMOKE_TEST else 2000,
    out_csv=str(OUT / f'{VARIANT}_per_generator.csv'),
)

In [ ]:
# Cell 10: Generate LOGO train/test manifests
# True LOGO requires RETRAINING on each train manifest (point a train notebook's
# cfg.dataset.metadata_paths at logo_train_wo_<gen>.csv), then evaluating that
# checkpoint on the matching logo_test_<gen>.csv with evaluate_per_generator.
from src.logo_eval import make_logo_manifests, FAKE_GENERATORS

logo_dir = PROJECT_ROOT / 'dataset' / 'metadata' / 'logo'
gens = FAKE_GENERATORS[:1] if SMOKE_TEST else FAKE_GENERATORS
made = {}
for g in gens:
    try:
        made[g] = make_logo_manifests(MANIFEST, g, out_dir=str(logo_dir), seed=SEED)
    except ValueError as e:
        print(f'skip {g}: {e}')
print(f'LOGO manifests written for {list(made)} -> {logo_dir}')

In [ ]:
# Cell 11: Summary of produced artifacts
for p in sorted(OUT.glob('*.csv')):
    print(p.relative_to(PROJECT_ROOT))
print('\nPAPER_EVALS COMPLETE - no errors')

In [ ]:
# Cell 12: Upload paper_evals results to HuggingFace
print('\nUploading paper_evals results to HuggingFace...')
mode = 'verify' if SMOKE_TEST else 'full_scale'
paper_evals_dir = PROJECT_ROOT / 'paper' / 'result' / mode / 'paper_evals'

if paper_evals_dir.exists():
    for f in paper_evals_dir.iterdir():
        if f.is_file() and f.suffix in ('.json', '.csv'):
            env.upload_to_hf(f, env.hf_results_repo, f'results/paper_evals/{f.name}')

print('Paper evals results upload complete')